In [69]:
import os
from pathlib import Path

import fiona
import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio as rio
import rasterio.mask as rio_mask

In [2]:
ghsl_path = Path(os.environ["GHSL_PATH"])

In [7]:
fiona.listlayers(ghsl_path / "GHS_UCDB_GLOBE_R2024A.gpkg")

['GHS_UCDB_THEME_CLIMATE_GLOBE_R2024A',
 'GHS_UCDB_THEME_EXPOSURE_GLOBE_R2024A',
 'GHS_UCDB_THEME_GENERAL_CHARACTERISTICS_GLOBE_R2024A',
 'GHS_UCDB_THEME_GEOGRAPHY_GLOBE_R2024A',
 'GHS_UCDB_THEME_GHSL_GLOBE_R2024A',
 'GHS_UCDB_THEME_GREENNESS_GLOBE_R2024A',
 'GHS_UCDB_THEME_HAZARD_RISK_GLOBE_R2024A',
 'GHS_UCDB_THEME_INFRASTRUCTURES_GLOBE_R2024A',
 'GHS_UCDB_THEME_LULC_GLOBE_R2024A',
 'GHS_UCDB_THEME_NATURAL_SYSTEMS_GLOBE_R2024A',
 'GHS_UCDB_THEME_SDG_GLOBE_R2024A',
 'GHS_UCDB_THEME_SOCIOECONOMIC_GLOBE_R2024A',
 'GHS_UCDB_THEME_WATER_GLOBE_R2024A',
 'GHS_UCDB_THEME_EMISSIONS_GLOBE_R2024A',
 'GHS_UCDB_THEME_HEALTH_GLOBE_R2024A',
 'UC_centroids']

In [49]:
wanted_countries = [
    "Argentina",
    "Bahamas",
    "Belize",
    "Bolivia",
    "Brazil",
    "Barbados",
    "Chile",
    "Colombia",
    "Costa Rica",
    "CostaRica",
    "Cuba",
    "Dominican Republic",
    "DominicanRepublic",
    "Ecuador",
    "El Salvador",
    "ElSalvador",
    "Guatemala",
    "French Guiana",
    "FrenchGuiana",
    "Guyana",
    "Honduras",
    "Haiti",
    "Jamaica",
    "Mexico",
    "México",
    "Nicaragua",
    "Panama",
    "Peru",
    "Paraguay",
    "Suriname",
    "Trinidad and Tobago",
    "TrinidadandTobago",
    "Uruguay",
    "Venezuela",
]

In [77]:
df_fua = (
    gpd.read_file(
        ghsl_path / "GHS_FUA_UCDB2015_GLOBE_R2019A_54009_1K_V1_0.gpkg",
        columns=["eFUA_name", "Cntry_name"],
    )
    .rename(columns={"eFUA_name": "fua_name", "Cntry_name": "country"})
    .loc[lambda df: df["country"].isin(wanted_countries)]
    .reset_index(drop=True)
)

df_ucdb = (
    gpd.read_file(
        ghsl_path / "GHS_UCDB_GLOBE_R2024A.gpkg",
        layer="GHS_UCDB_THEME_GENERAL_CHARACTERISTICS_GLOBE_R2024A",
        columns=["GC_UCN_MAI_2025", "GC_CNT_GAD_2025"],
    )
    .rename(
        columns={"GC_UCN_MAI_2025": "urban_center_name", "GC_CNT_GAD_2025": "country"},
    )
    .loc[lambda df: df["country"].isin(wanted_countries)]
    .reset_index(drop=True)
)

In [71]:
def add_total_pop(
    polygons: gpd.GeoDataFrame,
) -> gpd.GeoDataFrame:
    out = []
    for year in range(1975, 2021, 5):
        raster_path = ghsl_path / "POP_1000" / f"{year}.tif"
        with rio.open(raster_path) as ds:
            for idx, geom in polygons["geometry"].items():
                masked, _ = rio_mask.mask(ds, [geom], crop=True, nodata=0)
                out.append(
                    {
                        "idx": idx,
                        "year": year,
                        "pop": masked.sum(),
                    },
                )

    pops = (
        pd.DataFrame(out)
        .pivot_table(index="idx", columns="year", values="pop")
        .add_prefix("pop_", axis=1)
    )
    return pd.concat([polygons, pops], axis=1).pipe(
        gpd.GeoDataFrame,
        geometry="geometry",
        crs=polygons.crs,
    )


def add_smod_pop(
    polygons: gpd.GeoDataFrame,
) -> gpd.GeoDataFrame:
    res = []
    for year in range(1975, 2021, 5):
        with (
            rio.open(ghsl_path / "POP_1000" / f"{year}.tif") as ds_pop,
            rio.open(ghsl_path / "SMOD_1000" / f"{year}.tif") as ds_smod,
        ):
            for idx, geom in polygons["geometry"].items():
                masked_pop, _ = rio_mask.mask(ds_pop, [geom], crop=True, nodata=0)
                masked_smod, _ = rio_mask.mask(ds_smod, [geom], crop=True, nodata=0)

                masked_pop = masked_pop.squeeze()
                masked_smod = (masked_smod // 10 * 10).squeeze()

                weighted_count = np.bincount(
                    masked_smod.reshape(-1),
                    weights=masked_pop.reshape(-1),
                )

                res.append(
                    {
                        "idx": idx,
                        "year": year,
                        "pop_urban_center": weighted_count[30]
                        if len(weighted_count) > 30
                        else 0,
                        "pop_urban_cluster": weighted_count[20]
                        if len(weighted_count) > 20
                        else 0,
                        "pop_rural": weighted_count[10]
                        if len(weighted_count) > 10
                        else 0,
                    },
                )

    concat = pd.DataFrame(res)
    df_urban_center = concat.pivot_table(
        index="idx",
        columns="year",
        values="pop_urban_center",
    ).add_prefix("pop_urban_center_")
    df_urban_cluster = concat.pivot_table(
        index="idx",
        columns="year",
        values="pop_urban_cluster",
    ).add_prefix("pop_urban_cluster_")
    df_rural = concat.pivot_table(
        index="idx",
        columns="year",
        values="pop_rural",
    ).add_prefix(
        "pop_rural_",
    )

    return pd.concat(
        [polygons, df_urban_center, df_urban_cluster, df_rural],
        axis=1,
    ).pipe(gpd.GeoDataFrame, geometry="geometry", crs=polygons.crs)


def add_area_and_densities(polygons: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    out = polygons.assign(
        area_km2=polygons.to_crs("ESRI:54009")["geometry"].area / 1e6,
    )
    for year in range(1975, 2021, 5):
        out = out.assign(
            **{f"density_{year}": lambda df: df[f"pop_{year}"] / df["area_km2"]},
        )

    return out

In [78]:
df_ucdb_pop = add_area_and_densities(add_smod_pop(add_total_pop(df_ucdb)))
df_fua_pop = add_area_and_densities(add_smod_pop(add_total_pop(df_fua)))

In [79]:
df_ucdb_pop.to_file("./urban_centers.gpkg")
df_fua_pop.to_file("./functional_urban_areas.gpkg")

In [80]:
df_ucdb_pop.drop(columns=["geometry"]).to_excel("./urban_centers.xlsx", index=False)
df_fua_pop.drop(columns=["geometry"]).to_excel(
    "./functional_urban_areas.xlsx",
    index=False,
)